# Nowcasting Autoencoder Tutorial

Creating an effective autoencoder is a great first step in developing a predictive model. Learning how to create a high-quality latent state is critical, and can later be used to train a new encoder for a pre-existing predictive model, or a new decoder. An example for this could be training an encoder which uses different or fewer variables than the original model. Another might be training a higher-resolution decoder which is capable of producing higher-resolution predictions from the same latent state.

Autoencoders are also a great concept to learn when understanding neural network architectures.

In this example, we train an autencoder to perform dimensionality reduction and produce a useful latent state which can be used to resonstruct the original inputs.

### Initial set up

Import libraries

In [1]:
import pathlib
import datetime

In [2]:
import xarray as xr
import matplotlib.pyplot as plt

In [3]:
import pyearthtools.data as petdata
import pyearthtools.pipeline as petpipe

from pyearthtools.data.time import Petdt
from pyearthtools.pipeline.operations.xarray.join import GeospatialTimeSeriesMerge


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim

If you are at a different site to NCI, then change this import as appropriate.

In [5]:
import site_archive_nci

Set up pytorch for use

In [6]:
# Set random seed for reproducibility
torch.manual_seed(42)

# Autodetect GPU and use if possible
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

## Load the data

In [7]:
rf3proj = petdata.transforms.projection.Rainfields3ProjAus()
radar_projector = petdata.transforms.projection.XYtoLonLatRectilinear(rf3proj)

We specify the date, hour, and minute for querying data

In [8]:
selected_date = '2022-03-01T0700'

In [9]:
radar = petdata.archive.Rainfields3(variables='prcp-crate')

In [10]:
radar[selected_date]

<xarray.Dataset> Size: 40MB
Dimensions:     (y: 2050, n2: 2, x: 2450)
Coordinates:
  * y           (y) float64 16kB -1.001e+03 -1.003e+03 ... -5.097e+03 -5.099e+03
  * x           (x) float64 20kB -2.299e+03 -2.297e+03 ... 2.597e+03 2.599e+03
Dimensions without coordinates: n2
Data variables:
    valid_time  datetime64[ns] 8B 2022-03-01T07:00:00
    proj        int8 1B 0
    y_bounds    (y, n2) float64 33kB -1e+03 -1.002e+03 ... -5.098e+03 -5.1e+03
    x_bounds    (x, n2) float64 39kB -2.3e+03 -2.298e+03 ... 2.598e+03 2.6e+03
    rain_rate   (y, x) float64 40MB nan nan nan nan nan ... nan nan nan nan nan
Attributes:
    Conventions:         CF-1.7
    contributing_sites:  [ 2  3  4  5  6  7  8  9 14 15 16 17 19 22 23 24 25 ...
    institution:         Commonwealth of Australia, Bureau of Meteorology (AB...
    licence:             http://www.bom.gov.au/other/copyright.shtml
    source:              rainfields 3.2.13 drs-rainfields 2022-01-24
    station_id:          310
    station_name:        Ausm310
    title:               Bias Corrected Rainfall Rate Mosaic

### Build a pipeline for radar data

In [11]:
radarpipe = petpipe.Pipeline(
    radar,
    radar_projector,
    petpipe.operations.xarray.metadata.Rename({'valid_time':'time'}),
)

In [13]:
radarpipe[selected_date]

<xarray.Dataset> Size: 1GB
Dimensions:    (longitude: 4500, latitude: 4000, n2: 2)
Coordinates:
    x          (longitude, latitude) float64 144MB -1.806e+03 ... 2.673e+03
    y          (longitude, latitude) float64 144MB -5.069e+03 ... -759.1
  * longitude  (longitude) float64 36kB 110.0 110.0 110.0 ... 155.0 155.0 155.0
  * latitude   (latitude) float64 32kB -45.0 -44.99 -44.98 ... -5.03 -5.02 -5.01
Dimensions without coordinates: n2
Data variables:
    time       datetime64[ns] 8B 2022-03-01T07:00:00
    proj       int8 1B 0
    y_bounds   (longitude, latitude, n2) float64 288MB -5.068e+03 ... nan
    x_bounds   (longitude, latitude, n2) float64 288MB -1.807e+03 ... nan
    rain_rate  (longitude, latitude) float64 144MB nan nan nan ... nan nan nan
Attributes:
    Conventions:         CF-1.7
    contributing_sites:  [ 2  3  4  5  6  7  8  9 14 15 16 17 19 22 23 24 25 ...
    institution:         Commonwealth of Australia, Bureau of Meteorology (AB...
    licence:             http://www.bom.gov.au/other/copyright.shtml
    source:              rainfields 3.2.13 drs-rainfields 2022-01-24
    station_id:          310
    station_name:        Ausm310
    title:               Bias Corrected Rainfall Rate Mosaic

In [16]:
dir(petpipe.operations.xarray)

['AlignDates',
 'Chunk',
 'Compute',
 'Concatenate',
 'Merge',
 'RecodeCalendar',
 'Sort',
 '__all__',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__path__',
 '__spec__',
 '_align_dates',
 '_recode_calendar',
 'chunk',
 'compute',
 'conversion',
 'filters',
 'join',
 'metadata',
 'normalisation',
 'remapping',
 'reshape',
 'select',
 'sort',
 'split',
 'values']

In [20]:
dir(pyearthtools.pipeline.operations.xarray)

['AlignDates',
 'Chunk',
 'Compute',
 'Concatenate',
 'Merge',
 'RecodeCalendar',
 'Sort',
 '__all__',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__path__',
 '__spec__',
 '_align_dates',
 '_recode_calendar',
 'chunk',
 'compute',
 'conversion',
 'filters',
 'join',
 'metadata',
 'normalisation',
 'remapping',
 'reshape',
 'select',
 'sort',
 'split',
 'values']

In [14]:
full = petpipe.Pipeline(
    radar,
    radar_projector,
    petpipe.operations.xarray.metadata.Rename({'valid_time':'time'}),
    petpipe.operations.xarray.Sort(order=['time', 'latitude', 'longitude']),  # 
    # Align the data variable's coordinate order to the dataset coordinate order so all arrays are the same shape
    petpipe.operations.xarray.AlignDataVariableDimensionsToDatasetCoords(),  
    petdata.transform.region.Bounding(-35, -25, 138, 150),  # cut down on region for example
    petpipe.operations.xarray.normalisation.SingleValueDivision(1200),
    petpipe.operations.xarray.conversion.ToNumpy(),
    petpipe.operations.numpy.reshape.Rearrange('c t h w -> t c h w'), # channel time height width -> time channel height width
    iterator=petpipe.iterators.DateRange('20200101T00', '20210101T00', interval='10 minutes'),
    exceptions_to_ignore=petdata.exceptions.DataNotFoundError,
)

AttributeError: module 'pyearthtools.pipeline.operations.xarray' has no attribute 'AlignDataVariableDimensionsToDatasetCoords'

In [ ]:
%%time

# Takes around 15 seconds per sample to retrieve, largely due to the zip compression used on-disk
merged_sample = next(ipipe)
merged_sample

In [ ]:
full = petpipe.Pipeline(
    (satpipe, radarpipe),
    GeospatialTimeSeriesMerge(reference_dataset=himawari_sample), # These are pretty similar grids, so just pick one
    petdata.transforms.variables.Drop(['x_bounds', 'y_bounds', 'proj', 'x', 'y']),
    petpipe.operations.xarray.Sort(order=['time', 'latitude', 'longitude']),  # 
    petpipe.operations.xarray.AlignDataVariableDimensionsToDatasetCoords(),  # Align data variables coordinate ordering to dataset coordinate ordering
    petdata.transform.region.Bounding(-40, -25, 135, 152),  # cut down on region for example
    petpipe.operations.xarray.conversion.ToNumpy(),
    petpipe.operations.numpy.reshape.Rearrange('c t h w -> t c h w'), # channel time height width -> time channel height width
    iterator=petpipe.iterators.DateRange('20200101T00', '20210101T00', interval='20 minutes')
)
full

ipipe = iter(full)  # Make an iterator to walk the time period

In [ ]:
fullsat = petpipe.Pipeline(
    satpipe,
    # GeospatialTimeSeriesMerge(reference_dataset=himawari_sample), # These are pretty similar grids, so just pick one
    # petdata.transforms.variables.Drop(['x_bounds', 'y_bounds', 'proj', 'x', 'y']),
    petpipe.operations.xarray.Sort(order=['time', 'latitude', 'longitude']),  # 
    # Align the data variable's coordinate order to the dataset coordinate order so all arrays are the same shape
    petpipe.operations.xarray.AlignDataVariableDimensionsToDatasetCoords(),  
    petdata.transform.region.Bounding(-35, -25, 138, 150),  # cut down on region for example
    petpipe.operations.xarray.normalisation.SingleValueDivision(1200),
    petpipe.operations.xarray.conversion.ToNumpy(),
    petpipe.operations.numpy.reshape.Rearrange('c t h w -> t c h w'), # channel time height width -> time channel height width
    iterator=petpipe.iterators.DateRange('20200101T00', '20210101T00', interval='10 minutes'),
    exceptions_to_ignore=petdata.exceptions.DataNotFoundError
)
fullsat

ipipe = iter(fullsat)  # Make an iterator to walk the time period

In [ ]:
fullsat.exceptions_to_ignore

In [ ]:
n = next(ipipe)
# n

In [ ]:
plt.imshow(n[0][0])

In [ ]:
# Here we define an "AutoEncoder". This is a model which reproduces its inputs, 
# through a bottleneck layer. It is one of the primary concepts behind many 
# neural network architectures that you will work with in future, and is key to
# conceptual understanding as well as being sometimes useful in 

# Reminder, the image size is latitude: 1726, longitude: 2214

class AutoEncoder(nn.Module):
    def __init__(self, 
                 input_height = 501,
                 input_width = 601,
                 kernel_size = 4,
                 stride=2,
                 input_channel_count = 2,
                 output_channel_count = 2,
                 latent_dim=300):
        super(AutoEncoder, self).__init__()

        self.input_width = input_width
        self.input_height = input_height
        self.input_channel_count = input_channel_count
        self.output_channel_count = output_channel_count

        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels=self.input_channel_count, out_channels=16, kernel_size=kernel_size, stride = stride, padding = 1),
            nn.ReLU(),
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride =2, padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=7),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(in_channels=64, out_channels=32, kernel_size=7),
            nn.ReLU(),
            nn.ConvTranspose2d(in_channels=32, out_channels=16, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(in_channels=16, out_channels=self.output_channel_count, kernel_size=kernel_size, stride=stride, padding=1, output_padding=1),
            nn.Sigmoid()
        )
        

    def forward(self, x):

        # Get latent representation
        latent = self.encoder(x)

        # Reconstruct input
        reconstructed = self.decoder(latent)

        return reconstructed

In [ ]:
# Initialize model and move to device
model = AutoEncoder(input_channel_count=1, output_channel_count=1).to(device)

# Loss function and optimizer
criterion = nn.L1Loss()
# criterion = nn.KLDivLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
x = torch.from_numpy(n).float().to(device)

In [ ]:
# This should make us a messy prediction from an untrained model in the dimension of the input
prediction = model.forward(x)

In [ ]:
image_numpy_for_display = prediction.to('cpu').detach().numpy()
# image_numpy_for_display

In [ ]:
plt.imshow(image_numpy_for_display[0][0])

In [ ]:
%%time

# 1000 samples is taking about 2 minutes
# It should be able to go much much much faster, but still we can test it and make progress
# Let's to 30 minutes of training, i.e. 15k samples

def train(debug=True, num_epochs=1, max_samples=10, print_per=20):
    # Training loop
    
    sample_ix = 0 
    
    
    for epoch in range(num_epochs):
        total_loss = 0    
        epoch_samples = 0
        ipipe = iter(fullsat)  # Make an iterator to walk the time period
    
        while True:
            try:
                sample = next(ipipe)
            except StopIteration:
                break  # advance the epoch loop
            except:
                pass # some samples are just missing
    
            sample_ix += 1
            epoch_samples += 1
            if epoch_samples % print_per == 0:
                print(epoch_samples)
            
            if sample_ix > max_samples:
                break

            if debug:
                print(sample_ix)
    
            x = torch.from_numpy(sample).float().to(device)   

            if torch.any(torch.isnan(x)):
                # Skip nan inputs, they break the training
                continue
                
            if debug:
                print("Input")
                print(x)
    
            optimizer.zero_grad()
    
            # Forward pass
            y = model.forward(x)

            if debug:
                print("prediction")
                print(y)
            
            loss = criterion(y, x)
            if debug:
                print("loss")
                print(loss)
    
            # Backward pass and optimize        
            loss.backward()
            optimizer.step()
    
            total_loss += loss.item()
    
        # Print epoch statistics
        avg_loss = total_loss / epoch_samples
        epoch_samples = 0  # Reset for next epoch
        print(f'Epoch [{epoch+1}/{epoch_samples}], Average Loss: {avg_loss:.4f}')

train(debug=False, num_epochs=1, max_samples=15 * 1000, print_per = 500)

In [ ]:
x.min()

In [ ]:
prediction = model.forward(x)
image_numpy_for_display = prediction.to('cpu').detach().numpy()
plt.imshow(image_numpy_for_display[0][0])

In [ ]:


z = torch.from_numpy(fullsat['20210303T0400']).float().to(device)

image_numpy_for_display = z.to('cpu').detach().numpy()
plt.imshow(image_numpy_for_display[0][0])

In [ ]:
latent = model.encoder(z)

In [ ]:
latent.shape

In [ ]:
reconstruction = model.forward(z)
image_numpy_for_display = prediction.to('cpu').detach().numpy()
plt.imshow(image_numpy_for_display[0][0])